# DMOSpeech 2: Novel-Algorithm Comparison
## Three candidate handover algorithms vs. the current proposal

**Thesis:** A Risk-Calibrated Adaptive Handover Method for Teacher-Guided Flow-Matching TTS Inference

Every policy below is calibrated with the same Learn-Then-Test procedure ($\alpha = 0.05$, $\delta = 0.10$) on the calibration split and evaluated **once** on the held-out test split, on the same 500 Seed-TTS speakers as every earlier test.

| ID | Policy | Idea |
|---|---|---|
| B0 | Fixed Step 8 | DMOSpeech 2's fixed handover (reference) |
| B1 / B2 | SNR gate → Step 4 / Step 5 | Simple baselines (B1 = proposal's SNR-only rule) |
| **P0 / P1** | **Current proposal: SNR + drift** | Frozen $\hat\lambda = 0.50$ (P0); $\lambda$ re-certified in this pipeline (P1) |
| **C1** | **Student-Probe Handover** | At step $k$ the student takes its real first step from the teacher's guess; if the two agree, hand over and **reuse that step** (0 extra cost); otherwise keep the teacher. C1a: probe only clean-gated prompts; C1b: probe every prompt |
| **C2** | **Joint handover + restart** | Choose the handover step $k$ **and** the student's restart level $j$ (j = 2 restarts at τ = 0.293 with 2 student steps instead of re-noising back to 0.076) |
| **C3** | **Risk-budgeted SNR-band schedule** | One handover step per SNR band (<15, 15–20, 20–30, ≥30 dB), chosen from all 256 schedules |

**Protocol** frozen in `novel_algorithms_preregistration.json` before analysis. Verdicts compare each candidate with **P0** on test: IMPROVES (≥ 0.5 NFE cheaper and safe), MATCHES, WORSE, FAILS_GUARANTEE, NOT_CERTIFIED.

**Cost of the run:**
- C1 and C3 reuse the existing failure labels (`gpu_grid_all1000.csv`): about 30–45 minutes on a T4.
- C2 needs new audio for restart j = 2, scored with Whisper large-v3, WavLM and UTMOS: about 2–4 extra hours. Set `RUN_C2 = False` in Step 5 to skip it.
- Both runs are resumable if Colab disconnects.

---
### Instructions
1. Set the runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Choose `RUN_C2` in Step 5, then **Runtime > Run all**.
3. The last cell downloads `DMOSpeech2_Novel_Algorithms_Results.zip` with every output.

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Clone Thesis Repository & Install Dependencies
If Colab shows **"Restart session"** after the installs, click it, then continue from **Step 3** (no need to re-run Step 2). Every later step re-enters the repository folder and re-reads its settings, so a restart is safe.

In [ ]:
import os
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git checkout main
!git pull origin main

# Same pinned dependencies as the 1,000-utterance GPU retest notebook
!pip install -q "numpy==2.2.3" "pandas==2.2.3"
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

import numpy as np, pandas as pd, numba
print(f"Verified: NumPy {np.__version__} | Pandas {pd.__version__} | Numba {numba.__version__} [OK]")
!git log --oneline -1

runner_path = 'Handover Scheduling/colab/run_novel_algorithm_comparison.py'
if not os.path.exists(runner_path):
    raise FileNotFoundError(f"{runner_path} is not on main yet. Push it to GitHub (or upload it to this path) before continuing.")

### Step 3: Download DMOSpeech 2 Checkpoints
`model_85000.pt` (4.0 GB, teacher + student) and `model_1500.pt` (270 MB, duration predictor) into `ckpts/`.

In [ ]:
%cd /content/Shonget-DMOSpeech2
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

### Step 4: Download the Seed-TTS test-en Dataset
Prompt and ground-truth audio for the same 500 speakers (Dev 100 / Calibration 200 / Test 200) used in every earlier test.

In [ ]:
import os
os.chdir('/content/Shonget-DMOSpeech2')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'

# Check if audio is missing or only a Git LFS pointer
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print("Resolving Git LFS binaries for the Seed-TTS evaluation dataset...")
    if os.path.exists('data/seed-tts-eval/.git'):
        !cd data/seed-tts-eval && git lfs install && git lfs pull
    if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
        print("Cleaning leftover pointer files and downloading via huggingface_hub...")
        !rm -rf data/seed-tts-eval
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir="data/seed-tts-eval")
    print("Dataset setup complete!")
else:
    print("Seed-TTS audio binaries verified!")

!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav

### Step 5: Choose the Run and Smoke-Test It (4 prompts, a few minutes)
`RUN_C2 = True` runs all three candidates (C2 adds new synthesis and scoring). `RUN_C2 = False` runs C1, C3 and the baselines only.
The smoke test writes to a **separate** folder and includes C2's scoring path when `RUN_C2 = True`.

In [ ]:
RUN_C2 = True     # False = skip the j = 2 synthesis (C2) for a faster run

import os, json
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/dmospeech2_results', exist_ok=True)
json.dump({'RUN_C2': RUN_C2}, open('/content/drive/MyDrive/dmospeech2_results/novel_run_config.json', 'w'))   # later steps read this
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
os.chdir('/content/Shonget-DMOSpeech2')
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
os.makedirs(persist, exist_ok=True)
_cfg = os.path.join(persist, 'novel_run_config.json')
RUN_C2 = json.load(open(_cfg))['RUN_C2'] if os.path.exists(_cfg) else True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner = '"Handover Scheduling/colab/run_novel_algorithm_comparison.py"'
out_dir = os.path.join(persist, 'novel_algorithms' if RUN_C2 else 'novel_algorithms_fast')
c2_flag = '' if RUN_C2 else '--skip_c2'
print(f"RUN_C2 = {RUN_C2} | results in Google Drive: {out_dir}")

smoke_dir = '/content/novel_smoke'
!rm -rf "{smoke_dir}"
!python {runner} --device {device} --n_pairs 4 {c2_flag} --output_dir "{smoke_dir}"

import pandas as pd
smoke = pd.read_csv(f'{smoke_dir}/novel_features_all1000.csv')
print(f"Smoke test OK: {len(smoke)} rows, {smoke.shape[1]} columns")
cols = ['utt_id', 'condition', 'snr_energy', 'drift_4', 'probe_4', 'probe_5'] + (['fail_k5j2', 'fail_k8j2'] if RUN_C2 else [])
display(smoke[cols])

### Step 6: Full Run (1,000 prompts) + Analysis
Freezes the protocol, extracts drift and student-probe signals (and C2's j = 2 audio and scores if enabled) for all 500 speakers × 2 conditions, then compares every policy.

**If Colab disconnects:** the machine is reset (repository, checkpoints and dataset are deleted), but the results so far are safe in Google Drive (`MyDrive/dmospeech2_results/`). Re-run Steps 1–4, skip the smoke test, then run this cell again: it continues from the last saved row.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
os.chdir('/content/Shonget-DMOSpeech2')
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
os.makedirs(persist, exist_ok=True)
_cfg = os.path.join(persist, 'novel_run_config.json')
RUN_C2 = json.load(open(_cfg))['RUN_C2'] if os.path.exists(_cfg) else True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner = '"Handover Scheduling/colab/run_novel_algorithm_comparison.py"'
out_dir = os.path.join(persist, 'novel_algorithms' if RUN_C2 else 'novel_algorithms_fast')
c2_flag = '' if RUN_C2 else '--skip_c2'
print(f"RUN_C2 = {RUN_C2} | results in Google Drive: {out_dir}")

!python {runner} --device {device} {c2_flag} --output_dir "{out_dir}"

### Step 7: View the Report

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
os.chdir('/content/Shonget-DMOSpeech2')
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
os.makedirs(persist, exist_ok=True)
_cfg = os.path.join(persist, 'novel_run_config.json')
RUN_C2 = json.load(open(_cfg))['RUN_C2'] if os.path.exists(_cfg) else True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner = '"Handover Scheduling/colab/run_novel_algorithm_comparison.py"'
out_dir = os.path.join(persist, 'novel_algorithms' if RUN_C2 else 'novel_algorithms_fast')
c2_flag = '' if RUN_C2 else '--skip_c2'
print(f"RUN_C2 = {RUN_C2} | results in Google Drive: {out_dir}")
from IPython.display import Markdown, display

rep = os.path.join(out_dir, 'NOVEL_ALGORITHMS_REPORT.md')
if os.path.exists(rep):
    with open(rep, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
    s = json.load(open(os.path.join(out_dir, 'novel_algorithms_summary.json')))
    print("Verdicts vs. P0:", {k: v.get('verdict_vs_P0', 'reference') for k, v in s['policies'].items()})
    print("Reproduction:", s['reproduction'])
else:
    print(f"No report yet at {rep}. Run Step 6 first.")

### Step 8: Zip All Outputs and Download
Packs the feature CSV, preregistration, summary JSON, report, the runner scripts and the repository commit into one ZIP.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
os.chdir('/content/Shonget-DMOSpeech2')
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
os.makedirs(persist, exist_ok=True)
_cfg = os.path.join(persist, 'novel_run_config.json')
RUN_C2 = json.load(open(_cfg))['RUN_C2'] if os.path.exists(_cfg) else True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner = '"Handover Scheduling/colab/run_novel_algorithm_comparison.py"'
out_dir = os.path.join(persist, 'novel_algorithms' if RUN_C2 else 'novel_algorithms_fast')
c2_flag = '' if RUN_C2 else '--skip_c2'
print(f"RUN_C2 = {RUN_C2} | results in Google Drive: {out_dir}")
import zipfile, subprocess, datetime

if not os.path.exists(out_dir):
    raise FileNotFoundError(f"No results at {out_dir}. Run Step 6 first (and Step 5 to choose RUN_C2).")
zip_name = 'DMOSpeech2_Novel_Algorithms_Results.zip'
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(os.path.join(out_dir, 'RUN_INFO.txt'), 'w') as f:
    f.write(f"repository_commit: {commit}\n")
    f.write(f"run_c2: {RUN_C2}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in (out_dir, '/content/novel_smoke'):
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for fname in files:
                    if fname.startswith('temp_'):
                        continue
                    full = os.path.join(root, fname)
                    zf.write(full, os.path.join(os.path.basename(folder), os.path.relpath(full, folder)))
    for script in ('run_novel_algorithm_comparison.py', 'run_signal_selection_validation.py'):
        zf.write(f'Handover Scheduling/colab/{script}', f'code/{script}')

print(f"Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)")
with zipfile.ZipFile(zip_name) as zf:
    for n in zf.namelist():
        print("  ", n)

!cp "{zip_name}" "{persist}/"
print(f"Also saved to Google Drive: {persist}/{zip_name}")
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f"Download manually from the Files panel: /content/Shonget-DMOSpeech2/{zip_name}")